# P09 — BERT: preentrenamiento de Transformers bidireccionales profundos para comprensión del lenguaje

## 1. Título y paper

**Paper:** *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*  
**Autoría:** Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova  
**Año y venue:** 2018 · arXiv:1810.04805 · NAACL-HLT 2019  
**Nivel:** L3 · **Motor:** `bert_mlm`  
**Ficha completa:** [`P09_bert`](../../papers/foundational/P09_bert/README.md)

**Hito:** Consolida el patrón preentrenar-y-ajustar: un mismo modelo base sirve para muchas tareas con un ajuste pequeño.

- [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- [ACL Anthology (NAACL 2019)](https://aclanthology.org/N19-1423/)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los modelos de lenguaje eran unidireccionales; para comprender una palabra hace falta el contexto de ambos lados, y entrenar bidireccionalmente con predicción del siguiente token es trivialmente degenerado.
2. Ejecutar una implementación mínima de la propuesta: Modelado de lenguaje enmascarado (MLM) más predicción de la siguiente oración (NSP), y ajuste fino de todo el modelo por tarea.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08
- Peters et al. (2018), ELMo


## 4. Intuición

Para adivinar una palabra tapada, un humano mira lo que hay antes **y** después. Un modelo que solo mira hacia atrás está renunciando a la mitad de la evidencia disponible.


## 5. Concepto mínimo

MLM: se enmascara ~15 % de los tokens y se predice cada uno usando ambos lados.

```text
L = − Σ_{t ∈ enmascarados} log p(x_t | x_contexto_izquierdo, x_contexto_derecho)
```

No se puede entrenar bidireccionalmente con «predice el siguiente token»: cada token se vería a sí mismo a través de las capas. Enmascarar es lo que hace legítima la bidireccionalidad.


## 6. Código explicado

El motor cuenta candidatos compatibles con contexto solo-izquierdo frente a contexto bidireccional.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('bert_mlm', seed=7)['result']
for fila in r['masked_predictions']:
    print(f"«{fila['izquierda']} [MASK] {fila['derecha']}» (gold: {fila['gold']})")
    print('   solo izquierda :', fila['candidatos_solo_izquierda'], f"({fila['ambiguedad_izquierda']} opciones)")
    print('   bidireccional  :', fila['candidatos_bidireccional'], f"({fila['ambiguedad_bidireccional']} opciones)")

## 7. Predicción antes de ejecutar

1. ¿Reducirá el contexto derecho el número de candidatos, o lo dejará igual?
2. En «el banco [MASK] estaba mojado», ¿qué sentido de «banco» activa el contexto derecho?
3. ¿Por qué NO se puede entrenar un modelo bidireccional prediciendo el token siguiente?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for fila in r['masked_predictions']:
    reduccion = fila['ambiguedad_izquierda'] - fila['ambiguedad_bidireccional']
    print(f"gold={fila['gold']:<8} · ambigüedad izq={fila['ambiguedad_izquierda']} "
          f"bi={fila['ambiguedad_bidireccional']} · reducción={reduccion}")

## 9. Salida interpretable

El contexto derecho reduce (o iguala) el conjunto de candidatos, nunca lo amplía: añadir evidencia solo puede restringir. Ese es, en una línea, el argumento del paper.


## 10. Comentario pedagógico

Cuidado con el anacronismo inverso: BERT **no** es un modelo generativo de propósito general. Es un codificador para tareas de comprensión. La familia que hoy llamamos «LLM» viene de la otra rama, la del decoder (P10).


## 11. Error o anti-patrón deliberado

Anti-patrón: usar BERT como generador de texto autorregresivo porque «es un Transformer».


In [ ]:
print('Uso incorrecto: pedirle a un encoder MLM que continúe un texto token a token.')
print('Su objetivo de entrenamiento nunca fue p(x_t | x_<t): rellena huecos, no continúa.')

## 12. Corrección

Elige la familia según el objetivo de preentrenamiento, no según la arquitectura:


In [ ]:
familias = {
    'encoder (BERT)': {'objetivo': 'MLM', 'bueno_para': 'clasificar, extraer, buscar, comparar'},
    'decoder (GPT)': {'objetivo': 'siguiente token', 'bueno_para': 'generar, completar, dialogar'},
    'encoder-decoder (T5, BART)': {'objetivo': 'denoising / seq2seq', 'bueno_para': 'traducir, resumir'},
}
show(familias)

## 13. Desafío guiado

Añade una frase nueva al corpus y comprueba cómo cambia la ambigüedad del hueco.


In [ ]:
corpus = [
    'el banco del parque estaba mojado por la lluvia'.split(),
    'el banco del rio estaba mojado por la lluvia'.split(),
    'el banco del museo estaba mojado por la lluvia'.split(),   # <-- frase nueva
]
candidatos = {}
for linea in corpus:
    i = linea.index('del') + 1
    candidatos[linea[i]] = candidatos.get(linea[i], 0) + 1
print('candidatos para «el banco del [MASK] estaba mojado»:', candidatos)

## 14. Desafío autónomo

Con un modelo BERT en español de acceso abierto y ejecución local, compara la probabilidad del token correcto con contexto completo y truncando el contexto derecho. Usa 20 frases propias y reporta la diferencia media, la desviación y los casos donde no ayuda.


## 15. Evidencia de aprendizaje

Guarda la tabla de ambigüedad, la justificación de por qué enmascarar habilita la bidireccionalidad, y la tabla de familias con su objetivo de preentrenamiento.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P09_bert/README.md) · evaluación formal: [`assessments/papers/P09_bert.md`](../../assessments/papers/P09_bert.md)


## 16. Cierre

El preentrenamiento ya es la norma. Falta descubrir qué ocurre cuando la rama del decoder se escala dos órdenes de magnitud.


## 17. Conexión con el siguiente hito

- P10

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
